In [1]:
# Angka tetap. Setiap undian acak di notebook ini memakai SEED.
SEED = 42


# Kunci latihan

Setiap sel jawaban mencetak hasilnya, lalu memeriksanya. Jalankan dari atas.


## Kunci latihan 1


In [2]:
# Pustaka adalah paket kode siap pakai yang kita panggil.
# Satu ide: jumlah dan rata-rata tiga gelas latih. Angka 70 tidak ikut.
import numpy as np

gelas = np.array([10, 30, 50, 70, 90, 110, 130, 150])
jumlah_latih = int(gelas[:3].sum())
rata_latih = float(gelas[:3].mean())
rata_semua = float(gelas[:4].mean())
print("jumlah_latih", jumlah_latih)
print("rata_latih", rata_latih)
print("rata_empat", rata_semua)
print("beda", rata_latih != rata_semua)
assert jumlah_latih == 90, "Jumlah tiga gelas latih harus 90."
assert rata_latih == 30.0, "Rata-rata tiga gelas latih harus 30.0."
assert rata_latih != rata_semua, "Rata-rata latih harus berbeda dari rata-rata empat gelas."


jumlah_latih 90
rata_latih 30.0
rata_empat 40.0
beda True


## Kunci latihan 2 dan 3


In [ ]:
# Satu ide: siapkan alat untuk tabel dan rantai.
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


In [ ]:
# Satu ide: susun tabel latihan. Coretan ditulis, tetapi belum masuk rantai.
suhu = ["panas", "dingin"] * 4
tabel = pd.DataFrame(
    {
        "gelas_ml": gelas,
        "suhu": suhu,
        "catatan_pinggir": "gelas",
        "kembali": ["ya" if nilai < 100 else "tidak" for nilai in gelas],
    }
)
X = tabel[["gelas_ml", "suhu", "catatan_pinggir"]]
y = tabel["kembali"]


In [4]:
# Satu ide: sisihkan seperempat baris sebagai data uji.
X_latih, X_uji, y_latih, y_uji = train_test_split(
    X, y, test_size=0.25, random_state=SEED
)


In [5]:
# Satu ide: susun rantai. Kolom coretan tidak disebut, jadi terbuang.
penggabung = ColumnTransformer(
    transformers=[
        ("angka", StandardScaler(), ["gelas_ml"]),
        ("kategori", OneHotEncoder(sparse_output=False), ["suhu"]),
    ],
    remainder="drop",
)
pipa = Pipeline(
    steps=[
        ("praolah", penggabung),
        ("model", LogisticRegression(random_state=SEED)),
    ]
)


In [6]:
# Satu ide: dua kali jalan memberi tebakan yang sama.
pipa.fit(X_latih, y_latih)
tebakan_1 = pipa.predict(X_uji)
pipa.fit(X_latih, y_latih)
tebakan_2 = pipa.predict(X_uji)
print("tebakan", tebakan_1.tolist())
print("sama", tebakan_1.tolist() == tebakan_2.tolist())
assert tebakan_1.tolist() == tebakan_2.tolist(), "Dua jalan rantai yang sama harus memberi tebakan yang sama."


tebakan ['ya', 'ya']
sama True


In [7]:
# Satu ide: coretan tidak ikut, dan banyak tebakan yang cocok adalah satu.
nama_fitur = pipa.named_steps["praolah"].get_feature_names_out().tolist()
print("nama_fitur", nama_fitur)
print("ada_catatan", any("catatan_pinggir" in nama for nama in nama_fitur))
cocok = int((tebakan_1 == y_uji.to_numpy()).sum())
print("cocok", cocok)
print("dari", len(y_uji))
assert not any("catatan_pinggir" in nama for nama in nama_fitur), (
    "Coretan tidak boleh ikut menjadi kolom."
)
assert cocok == 1, "Banyak tebakan yang cocok pada kunci harus 1."
assert len(y_uji) == 2, "Data uji kunci harus dua baris."


nama_fitur ['angka__gelas_ml', 'kategori__suhu_dingin', 'kategori__suhu_panas']
ada_catatan False
cocok 1
dari 2


## Kunci tambahan: kategori yang belum pernah dilihat


In [4]:
# Nilai bawaan OneHotEncoder menolak kategori yang tidak ada saat belajar.
# ValueError adalah tanda bahwa isi yang masuk tidak diterima.
# Satu ide: coba tulisan "hangat", yang tidak ada di data latih.
pengkode = pipa.named_steps["praolah"].named_transformers_["kategori"]
gagal = False
try:
    pengkode.transform(pd.DataFrame({"suhu": ["hangat"]}))
except ValueError:
    gagal = True
print("error", gagal)
assert gagal, "Kategori yang tidak dikenal harus ditolak."


error True
